# Phase 7: Snowpark Python on Snowflake

Learn Snowpark by querying SALES_DW data using Python DataFrames, creating UDFs, and building stored procedures.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType, FloatType, IntegerType

session = get_active_session()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")

In [ ]:
customers_df = session.table('SALES_DW.STAGING.STG_CUSTOMERS')
print(f"Columns: {customers_df.columns}")
print(f"Row count: {customers_df.count()}")
customers_df.show(5)

In [ ]:
premium_customers = customers_df.filter(F.col('SEGMENT') == 'PREMIUM')
print(f"Premium customers: {premium_customers.count()}")
premium_customers.select('CUSTOMER_NAME', 'EMAIL', 'COUNTRY').show()

In [ ]:
segment_stats = customers_df.group_by('SEGMENT').agg(
    F.count('CUSTOMER_ID').alias('CUSTOMER_COUNT')
).sort('CUSTOMER_COUNT', ascending=False)

segment_stats.show()

In [ ]:
orders_df = session.table('SALES_DW.STAGING.STG_ORDERS')

customer_orders = customers_df.join(
    orders_df,
    customers_df['CUSTOMER_ID'] == orders_df['CUSTOMER_ID']
).select(
    customers_df['CUSTOMER_NAME'],
    customers_df['SEGMENT'],
    customers_df['COUNTRY'],
    orders_df['ORDER_ID'],
    orders_df['ORDER_DATE'],
    orders_df['TOTAL_AMOUNT']
)

customer_orders.sort(F.col('TOTAL_AMOUNT').desc()).show(10)

In [ ]:
top_customers = customers_df.join(
    orders_df,
    customers_df['CUSTOMER_ID'] == orders_df['CUSTOMER_ID']
).group_by(
    customers_df['CUSTOMER_NAME'],
    customers_df['SEGMENT'],
    customers_df['COUNTRY']
).agg(
    F.count('ORDER_ID').alias('TOTAL_ORDERS'),
    F.sum('TOTAL_AMOUNT').alias('TOTAL_SPENT')
).sort(F.col('TOTAL_SPENT').desc())

top_customers.show(10)

In [ ]:
import matplotlib.pyplot as plt

segment_pd = segment_stats.to_pandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(segment_pd['SEGMENT'], segment_pd['CUSTOMER_COUNT'], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_title('Customers by Segment')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
session.sql('CREATE STAGE IF NOT EXISTS SALES_DW.ANALYTICS.UDF_STAGE').collect()

@F.udf(name='SALES_DW.ANALYTICS.CUSTOMER_VALUE_SCORE',
       return_type=StringType(),
       input_types=[FloatType(), IntegerType()],
       is_permanent=True,
       stage_location='@SALES_DW.ANALYTICS.UDF_STAGE',
       replace=True,
       packages=['snowflake-snowpark-python'])
def customer_value_score(total_spent: float, total_orders: int) -> str:
    if total_spent is None or total_orders is None:
        return 'UNKNOWN'
    if total_spent > 500 and total_orders >= 3:
        return 'PLATINUM'
    elif total_spent > 200 or total_orders >= 2:
        return 'GOLD'
    elif total_spent > 0:
        return 'SILVER'
    else:
        return 'BRONZE'

print('UDF CUSTOMER_VALUE_SCORE created!')

In [ ]:
%%sql -r scored_customers
SELECT
    CUSTOMER_NAME, SEGMENT, TOTAL_ORDERS, LIFETIME_REVENUE,
    SALES_DW.ANALYTICS.CUSTOMER_VALUE_SCORE(LIFETIME_REVENUE, TOTAL_ORDERS) AS VALUE_TIER
FROM SALES_DW.ANALYTICS.DIM_CUSTOMERS
ORDER BY LIFETIME_REVENUE DESC LIMIT 15;

In [ ]:
from snowflake.snowpark import Session

def data_quality_check(session: Session) -> str:
    checks = []
    null_emails = session.table('SALES_DW.STAGING.STG_CUSTOMERS').filter(F.col('EMAIL').is_null()).count()
    checks.append(f"Null emails: {null_emails}")
    orphan_orders = session.sql("""
        SELECT COUNT(*) as cnt FROM SALES_DW.STAGING.STG_ORDERS o
        WHERE NOT EXISTS (SELECT 1 FROM SALES_DW.STAGING.STG_CUSTOMERS c WHERE c.CUSTOMER_ID = o.CUSTOMER_ID)
    """).collect()[0][0]
    checks.append(f"Orphan orders: {orphan_orders}")
    neg_prices = session.table('SALES_DW.STAGING.STG_ORDER_ITEMS').filter(F.col('UNIT_PRICE') < 0).count()
    checks.append(f"Negative prices: {neg_prices}")
    total_issues = null_emails + orphan_orders + neg_prices
    status = 'PASS' if total_issues == 0 else 'FAIL'
    return f"Data Quality Report | Status: {status}\n" + "\n".join(checks)

result = data_quality_check(session)
print(result)

In [ ]:
session.sproc.register(
    func=data_quality_check,
    name='SALES_DW.ANALYTICS.DATA_QUALITY_CHECK',
    return_type=StringType(),
    is_permanent=True,
    stage_location='@SALES_DW.ANALYTICS.UDF_STAGE',
    replace=True,
    packages=['snowflake-snowpark-python']
)
print('Stored Procedure DATA_QUALITY_CHECK registered!')

In [ ]:
%%sql -r dq_result
CALL SALES_DW.ANALYTICS.DATA_QUALITY_CHECK();